In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()))

from environment import Environment
from collections import defaultdict, Counter
import pandas as pd
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle


csv_path = "baselines/observable_samples.csv"

### Generate samples

In [ ]:
SEED = 10
random.seed(SEED)
np.random.seed(SEED)

N = 500000
BATCH_SIZE = 10000

environment = Environment()
header_written = os.path.exists(csv_path)

batch = defaultdict(list)

IMPACT_THRESHOLDS = {
    "none":   0.0447,
    "yellow": 0.0673,
    "amber":  0.1657,
}

def get_impact_level(impact):
    if impact < IMPACT_THRESHOLDS["none"]:
        return 0  # no warning
    elif impact < IMPACT_THRESHOLDS["yellow"]:
        return 1  # yellow
    elif impact < IMPACT_THRESHOLDS["amber"]:
        return 2  # amber
    else:
        return 3  # red

for i in range(N):
    # --------------- Sampling
    environment.sample_features()
    environment.update_derived()

    # --------------- Saving samples
    features = environment.get_observable_features()
    impact = environment.impact
    warning_level = get_impact_level(impact)

    for key, value in features.items():
        batch[key].append(value)
    batch["warning_level"].append(warning_level)

    # --------------- Flush to disk every BATCH_SIZE iterations
    if (i + 1) % BATCH_SIZE == 0:
        print(f"Batch {(i + 1) // BATCH_SIZE}")

        df = pd.DataFrame(batch)
        df.to_csv(csv_path, mode="a", index=False, header=not header_written)
        header_written = True
        batch = defaultdict(list)

# Write any remaining rows
if batch:
    df = pd.DataFrame(batch)
    df.to_csv(csv_path, mode="a", index=False, header=not header_written)

### Visualise samples

In [ ]:
df = pd.read_csv(csv_path)

# calculate grid size based on number of columns
n = len(df.columns)
ncols = math.ceil(math.sqrt(n))
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()

for ax, col in zip(axes, df.columns):
    ax.hist(df[col].dropna(), bins=30)
    ax.set_title(col, fontsize=8)

# hide unused axes
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
warning_counts = df["warning_level"].value_counts(normalize=True).sort_index()
print("Warning level distribution:")
for level, pct in warning_counts.items():
    print(f"Level {level}: {pct:.2%}")

In [ ]:
# plot total warning levels
plt.figure(figsize=(6, 4))
plt.bar(warning_counts.index, warning_counts.values, color=["grey", "yellow", "orange", "red"])
plt.xticks(warning_counts.index, ["none", "yellow", "amber", "red"])
plt.xlabel("Warning Level") 
plt.ylabel("Proportion of Samples")
plt.title("Distribution of Warning Levels")

### Fit decision tree

In [2]:
SEED = 1
MAX_DEPTH = 6

df = pd.read_csv(csv_path)

season_map = {"Spring": 0, "Summer": 1, "Autumn": 2, "Winter": 3}
df["season"] = df["season"].map(season_map)
df["season_sin"] = np.sin(2 * np.pi * df["season"] / 4)
df["season_cos"] = np.cos(2 * np.pi * df["season"] / 4)
df = df.drop(columns=["season"])

X = df.drop(columns=["warning_level"])
y = df["warning_level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

print("Training set counts:", Counter(y_train))

sampling_strategy_over = {
    1: 100000,
    2: 50000,
    3: 10000,
}

sampling_strategy_under = {
    0: 200000,
}

over = SMOTE(sampling_strategy=sampling_strategy_over, random_state=SEED, k_neighbors=5)
under = RandomUnderSampler(sampling_strategy=sampling_strategy_under, random_state=SEED)

pipeline = Pipeline(steps=[("over", over), ("under", under)])
X_train_res, y_train_res = pipeline.fit_resample(X_train, y_train)

print("Resampled class distribution:")
unique, counts = np.unique(y_train_res, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")

clf = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=SEED)
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["none", "yellow", "amber", "red"], zero_division=0))

print("\nDecision Tree Rules:")
print(export_text(clf, feature_names=list(X.columns)))

with open("baselines/dt_model.pkl", "wb") as f:
    pickle.dump(clf, f)
print("Model saved to baselines/dt_model.pkl")

Training set counts: Counter({0: 284457, 1: 82094, 2: 30507, 3: 2942})
Resampled class distribution:
  0: 200000
  1: 100000
  2: 50000
  3: 10000

Classification Report:
              precision    recall  f1-score   support

        none       0.77      0.91      0.83     71114
      yellow       0.32      0.15      0.21     20524
       amber       0.46      0.30      0.36      7627
         red       0.60      0.77      0.67       735

    accuracy                           0.71    100000
   macro avg       0.54      0.53      0.52    100000
weighted avg       0.65      0.71      0.67    100000


Decision Tree Rules:
|--- flood_depth <= 0.00
|   |--- deprivation_index <= 0.47
|   |   |--- precipitation <= 1.03
|   |   |   |--- deprivation_index <= 0.28
|   |   |   |   |--- population_density <= 1661.16
|   |   |   |   |   |--- water_density <= 0.02
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- water_density >  0.02
|   |   |   |   |   |   |--- class: 0
|   |   |   |